# Diabetes 130-US Hospitals — Exploratory Analysis

**ALK Kozminski University — Semester Project 2**

This notebook provides interactive exploration of the dataset. Run the pipeline scripts first:
```
python scripts/01_ingest.py
python scripts/02_load.py
```

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine

print('Imports OK')

In [ ]:
# Load configuration
config_path = project_root / 'config.json'
with open(config_path) as f:
    config = json.load(f)

print('Config loaded:')
print(json.dumps(config['pipeline'], indent=2))

In [ ]:
# Connect to database
db_path = project_root / config['database']['path']
if not db_path.exists():
    print(f'Database not found at {db_path}')
    print('Run: python scripts/01_ingest.py && python scripts/02_load.py')
else:
    engine = create_engine(f'sqlite:///{db_path}', echo=False)
    print(f'Connected to: {db_path}')

## 1. Dataset Overview

In [ ]:
from scripts.query_helpers import summary_stats

stats = summary_stats(engine)
print('Table row counts:')
display(stats['row_counts'])

print('\nReadmission distribution:')
display(stats['readmission_dist'])

In [ ]:
print('Age group distribution:')
display(stats['age_dist'])

print('\nLength of stay stats:')
display(stats['los_stats'])

## 2. Readmission Analysis

In [ ]:
from scripts.query_helpers import readmission_by_group

df_age = readmission_by_group(engine, group_col='age_group')
display(df_age.head(20))

In [ ]:
# Readmission rate by age group
df_readmitted = df_age[df_age['readmission'].isin(['<30', '>30'])].copy()
total = df_age.groupby('group_value')['count'].sum().reset_index()
total.columns = ['group_value', 'total']
readmit_sum = df_readmitted.groupby('group_value')['count'].sum().reset_index()
readmit_sum.columns = ['group_value', 'readmitted']
merged = total.merge(readmit_sum, on='group_value', how='left').fillna(0)
merged['rate'] = merged['readmitted'] / merged['total'] * 100
merged = merged.sort_values('group_value')

fig = px.bar(
    merged, x='group_value', y='rate',
    title='Readmission Rate (%) by Age Group',
    labels={'group_value': 'Age Group', 'rate': 'Readmission Rate (%)'},
    color='rate', color_continuous_scale='viridis'
)
fig.show()

## 3. HbA1c Analysis

In [ ]:
from scripts.query_helpers import hba1c_vs_readmission

df_hba = hba1c_vs_readmission(engine)

fig = px.bar(
    df_hba, x='hba1c_result', y='rate', color='readmission', barmode='group',
    title='HbA1c Result vs Readmission Outcome',
    labels={'hba1c_result': 'HbA1c Result', 'rate': 'Proportion'}
)
fig.show()

## 4. Top Diagnoses

In [ ]:
from scripts.query_helpers import top_diagnoses_by_readmission

df_diag = top_diagnoses_by_readmission(engine, top_n=10)

totals = df_diag.groupby('icd9_code')['count'].sum().reset_index()
totals.columns = ['icd9_code', 'total']
readmitted = df_diag[df_diag['readmission'].isin(['<30', '>30'])].groupby('icd9_code')['count'].sum().reset_index()
readmitted.columns = ['icd9_code', 'readmitted']
diag_merged = totals.merge(readmitted, on='icd9_code', how='left').fillna(0)
diag_merged['rate'] = diag_merged['readmitted'] / diag_merged['total'] * 100
diag_merged = diag_merged.sort_values('rate')

fig = px.bar(
    diag_merged, x='rate', y='icd9_code', orientation='h',
    title='Top 10 Diagnoses by Readmission Rate',
    labels={'rate': 'Readmission Rate (%)', 'icd9_code': 'ICD-9 Code'}
)
fig.show()

## 5. Medications vs Length of Stay

In [ ]:
from scripts.query_helpers import medications_vs_los
from scipy import stats as scipy_stats

df_med = medications_vs_los(engine)
x = df_med['num_medications'].values.astype(float)
y = df_med['mean_los'].values.astype(float)

slope, intercept, r_value, p_value, _ = scipy_stats.linregress(x, y)
y_pred = slope * x + intercept

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Data',
                          marker=dict(size=8, color=x, colorscale='viridis')))
fig.add_trace(go.Scatter(x=x, y=y_pred, mode='lines', name=f'r²={r_value**2:.3f}',
                          line=dict(color='red', dash='dash')))
fig.update_layout(
    title='Number of Medications vs Mean Length of Stay',
    xaxis_title='Number of Medications',
    yaxis_title='Mean LOS (days)'
)
fig.show()
print(f'Pearson r² = {r_value**2:.4f}, p-value = {p_value:.4e}')

## 6. Parameter Sensitivity Example

In [ ]:
# Load raw CSV to demonstrate null threshold effect
raw_csv = project_root / config['data']['raw_csv']
if raw_csv.exists():
    df_raw = pd.read_csv(raw_csv, low_memory=False).replace('?', np.nan)
    null_pct = df_raw.isnull().mean().sort_values(ascending=False)
    
    print('Null percentage per column (top 10):')
    display(null_pct.head(10).to_frame('null_pct'))
    
    for threshold in [0.1, 0.3, 0.5]:
        n_dropped = (null_pct > threshold).sum()
        print(f'  null_threshold={threshold}: {n_dropped} columns would be dropped')
else:
    print('Raw CSV not found. Skipping sensitivity analysis.')